# 05 — Tuning Bayesiano con Optuna

Busca los mejores hiperparámetros para **HistGradientBoosting** usando Optuna (muestreo TPE).  
Cada trial se registra en MLflow. Al final se compara contra el **LogReg baseline** (PR-AUC OOF = 0.861).

**Criterio de selección de nuevo campeón:**  
El modelo tuneado reemplaza a LogReg solo si el IC 95% (bootstrap) de la diferencia `hist_gb_tuned − logreg` **excluye el 0** por la derecha.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import mlflow

from src.tuning import run_study, evaluate_best_trial
from src.evaluation import paired_bootstrap_diff
from src.data import load_train
from src import config as C
from src.mlflow_utils import setup_tracking

# Cargar baseline LogReg
with open(ROOT / 'reports' / 'numeros_oficiales.json', encoding='utf-8') as f:
    nums = json.load(f)

LOGREG_PR_AUC = nums['modelos_oof']['logreg']['pr_auc_media']
print(f'LogReg baseline PR-AUC (OOF): {LOGREG_PR_AUC:.4f}')
print(f'Optuna version: {optuna.__version__}')

## 1. Lanzar el estudio Optuna

> ⚠️ **Este paso tarda** (50 trials × CV = varios minutos).  
> Podés bajar `n_trials=20` para una prueba rápida.

In [ ]:
setup_tracking()  # sqlite:///mlflow.db

study = run_study(
    n_trials=50,          # aumentar a 100 para búsqueda más fina
    show_progress=True,
)

print(f'\nMejor trial: #{study.best_trial.number}')
print(f'  PR-AUC (CV rápida): {study.best_trial.value:.4f}')
print(f'  Parámetros:')
for k, v in study.best_trial.params.items():
    print(f'    {k}: {v}')

## 2. Visualizar la búsqueda

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Historia de trials
trial_values = [t.value for t in study.trials if t.value is not None]
best_so_far  = [max(trial_values[:i+1]) for i in range(len(trial_values))]

axes[0].plot(trial_values, 'o', alpha=0.5, color='#457b9d', ms=5, label='PR-AUC trial')
axes[0].plot(best_so_far, '-', color='#E63946', lw=2, label='Mejor acumulado')
axes[0].axhline(LOGREG_PR_AUC, color='green', lw=1.5, linestyle='--',
                label=f'LogReg baseline ({LOGREG_PR_AUC:.3f})')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('PR-AUC (CV)')
axes[0].set_title('Historia de la búsqueda Optuna')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Importancia de hiperparámetros
try:
    importance = optuna.importance.get_param_importances(study)
    names = list(importance.keys())
    vals  = list(importance.values())
    axes[1].barh(names, vals, color='#2A9D8F', edgecolor='white')
    axes[1].set_xlabel('Importancia relativa')
    axes[1].set_title('Importancia de hiperparámetros')
    axes[1].grid(axis='x', alpha=0.3)
except Exception as e:
    axes[1].text(0.5, 0.5, f'No disponible:\n{e}', ha='center', va='center',
                 transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / '08_optuna_search.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figura guardada en reports/figures/08_optuna_search.png')

## 3. Evaluar el mejor trial con CV completa (5×5)

La CV rápida del objetivo usa los mismos folds pero es más ruidosa. Ahora evaluamos con la CV completa para comparar justamente con LogReg.

In [ ]:
print('Evaluando mejor HistGB con CV completa (5×5)... puede tardar ~2 min')
results = evaluate_best_trial(study)

print(f"\n{'Métrica':<20} {'HistGB tuneado':>15} {'LogReg baseline':>15}")
print('-' * 52)
print(f"{'PR-AUC (OOF)':<20} {results['pr_auc_media']:>15.4f} {nums['modelos_oof']['logreg']['pr_auc_media']:>15.4f}")
print(f"{'ROC-AUC (OOF)':<20} {results['roc_auc_media']:>15.4f} {nums['modelos_oof']['logreg']['roc_auc_media']:>15.4f}")
print(f"{'PR-AUC SD':<20} {results['pr_auc_sd']:>15.4f} {nums['modelos_oof']['logreg']['pr_auc_sd']:>15.4f}")

## 4. Test de significancia (bootstrap pareado)

In [ ]:
import joblib
from sklearn.metrics import average_precision_score
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from src.pipelines.preprocessor import build_pipeline, n_categorical
from src.evaluation import get_cv

X, y, _ = load_train()

# OOF del LogReg (re-calculados con mismos folds)
logreg_pipe = joblib.load(ROOT / 'models' / 'dropout_logreg_v0.1-tp1.joblib')

from src.evaluation import cross_validate_oof
print('Calculando OOF de LogReg... (necesario para bootstrap pareado)')
_, logreg_oof = cross_validate_oof(logreg_pipe, X.drop(columns=[C.ID_COL]), y)

# Bootstrap pareado
diff, ci_lo, ci_hi = paired_bootstrap_diff(
    y.to_numpy(),
    results['oof_preds'],  # HistGB tuneado
    logreg_oof,            # LogReg
)

print(f'\nDiferencia HistGB_tuneado − LogReg:')
print(f'  Punto:  {diff:+.4f}')
print(f'  IC 95%: [{ci_lo:+.4f}, {ci_hi:+.4f}]')

if ci_lo > 0:
    print('\n✅ HistGB tuneado es MEJOR que LogReg (IC excluye 0 por la derecha)')
    print('   → Candidato a nuevo campeón para TF1')
elif ci_hi < 0:
    print('\n❌ LogReg es MEJOR que HistGB tuneado')
    print('   → LogReg se mantiene como modelo final')
else:
    print('\n⚖️  Empate estadístico (IC incluye 0)')
    print('   → LogReg se mantiene por parsimonia e interpretabilidad (ADR-08)')

## 5. Registrar resultado en MLflow

In [ ]:
from src.mlflow_utils import init_run, log_cv_results, log_threshold_metrics

cv_summary = {
    'pr_auc_media':  results['pr_auc_media'],
    'pr_auc_sd':     results['pr_auc_sd'],
    'roc_auc_media': results['roc_auc_media'],
}

with init_run(experiment='dropout_tf1', run_name='hist_gb_best_cv_completa',
              tags={'fase': 'tf1', 'modelo': 'hist_gb_tuneado'}) as run:
    log_cv_results(cv_summary)
    for k, v in results['best_params'].items():
        mlflow.log_param(k, v)
    mlflow.log_metric('bootstrap_diff_vs_logreg', diff)
    mlflow.log_metric('bootstrap_ci_lo', ci_lo)
    mlflow.log_metric('bootstrap_ci_hi', ci_hi)
    log_threshold_metrics(
        threshold=results['umbral'],
        recall=results['recall_oof'], precision=results['precision_oof'],
        f1=0.0, tp=0, fp=0, fn=0, tn=0,
    )
    print(f'✓ Resultado registrado en MLflow (run_id={run.info.run_id[:8]}…)')